### 질문 6 전처리

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 데이터 경로 설정 (환경에 맞게 수정)
DATA_DIR = r'C:\codeit\프로젝트1'


station = os.path.join(DATA_DIR, '서울시 역사마스터 정보.csv')
hospital = os.path.join(DATA_DIR, '병원정보서비스(2025.12.).xlsx')
nursing = os.path.join(DATA_DIR, '국민건강보험공단_장기요양기관 시설별 현황_20250401.xlsx')
welfare = os.path.join(DATA_DIR, '보건복지부_장애인복지관 현황_20240425.csv')
priority = os.path.join(DATA_DIR, 'priority_last.xlsx')
borough = os.path.join(DATA_DIR, '서울교통공사_자치구별지하철역정보_20260212.CSV')

In [3]:
station_df = pd.read_csv(station, encoding='cp949')
hospital_df = pd.read_excel(hospital)
nursing_df = pd.read_excel(nursing)
welfare_df  = pd.read_csv(welfare, encoding='cp949')
priority_df = pd.read_excel(priority)
borough_df = pd.read_csv(borough)

In [4]:
#station_df.info()
#hospital_df.info()
#nursing_df.info()
#welfare_df.info()
#priority_df.info()
borough_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   자치구      30 non-null     object
 1   해당역(호선)  30 non-null     object
 2   역개수      30 non-null     int64 
dtypes: int64(1), object(2)
memory usage: 852.0+ bytes


In [5]:
# 1. 서울 필터링

# hospital_df - 시도코드명 기준
hospital_seoul = hospital_df[hospital_df['시도코드명'] == '서울'].copy()

# nursing_df - 시도 시군구 법정동명이 '서울'로 시작
nursing_seoul = nursing_df[nursing_df['시도 시군구 법정동명'].str.startswith('서울', na=False)].copy()

# welfare_df - 시도 기준
welfare_seoul = welfare_df[welfare_df['시도'] == '서울'].copy()


# 확인
print(f"hospital_seoul: {len(hospital_seoul)}개")
print(f"nursing_seoul: {len(nursing_seoul)}개")
print(f"welfare_seoul: {len(welfare_seoul)}개")

hospital_seoul: 19641개
nursing_seoul: 3990개
welfare_seoul: 51개


In [6]:
# 2.교통약자 관련 병원 종류만 필터링
target_types = ['요양병원', '종합병원', '상급종합']
hospital_filtered = hospital_seoul[hospital_seoul['종별코드명'].isin(target_types)].copy()

In [7]:
# 확인
print(f"서울 전체: {len(hospital_seoul)}개")
print(f"필터링 후: {len(hospital_filtered)}개")
print(hospital_filtered['종별코드명'].value_counts())

서울 전체: 19641개
필터링 후: 162개
종별코드명
요양병원    103
종합병원     45
상급종합     14
Name: count, dtype: int64


In [8]:
# 3. 구별 병원 수 집계

hospital_by_gu = (
    hospital_filtered
    .groupby('시군구코드명')
    .size()
    .reset_index(name='병원수')
    .rename(columns={'시군구코드명': '구'})
)
print(hospital_by_gu.sort_values('병원수', ascending=False))

       구  병원수
19  영등포구   16
0    강남구   10
8    노원구   10
17   송파구   10
16   성북구    9
3    강서구    9
6    구로구    9
10  동대문구    9
1    강동구    8
21   은평구    7
9    도봉구    7
13  서대문구    7
2    강북구    6
14   서초구    6
22   종로구    5
18   양천구    5
24   중랑구    5
15   성동구    5
7    금천구    4
5    광진구    4
11   동작구    3
4    관악구    3
12   마포구    3
20   용산구    1
23    중구    1


In [9]:
# 4. priority_df의 역명으로 station_df 필터링
target_stations = priority_df['지하철역'].tolist()
target_df = station_df[station_df['역사명'].isin(target_stations)].copy()

print(f"매칭된 역 수: {len(target_df)}개")
print(target_df[['역사명', '호선', '위도', '경도']])

매칭된 역 수: 49개
         역사명          호선         위도          경도
5         서울  수도권 광역급행철도  37.555690  126.972960
6        연신내  수도권 광역급행철도  37.618780  126.921300
78        신림         신림선  37.484927  126.929616
98        강남        신분당선  37.496837  127.028104
113     홍대입구     공항철도1호선  37.557438  126.926715
242       암사         8호선  37.550210  127.127562
293       하계         7호선  37.636352  127.067990
327       망원         6호선  37.556094  126.910052
334      연신내         6호선  37.618636  126.920625
338       응암         6호선  37.598605  126.915577
361      장한평         5호선  37.561440  127.064623
370     종로3가         5호선  37.572540  126.990305
387       화곡         5호선  37.541513  126.840461
388      우장산         5호선  37.548768  126.836318
389       발산         5호선  37.558598  126.837668
605     홍대입구       경의중앙선  37.557641  126.926683
638       선릉         분당선  37.504856  127.048807
653      신도림         경부선  37.508787  126.891144
661       사당         4호선  37.476955  126.981651
670       명동         4호선  3

In [10]:
# 호선 우선순위: 1~9호선 숫자 호선 먼저
target_df['호선_우선순위'] = target_df['호선'].str.match(r'^\d').astype(int)

target_df_unique = (
    target_df
    .sort_values('호선_우선순위', ascending=False)  # 숫자 호선 우선
    .drop_duplicates(subset='역사명', keep='first')
    .copy()
)

print(f"중복 제거 후 역 수: {len(target_df_unique)}개")
print(target_df_unique[['역사명', '호선', '위도', '경도']].sort_values('역사명'))

중복 제거 후 역 수: 36개
         역사명          호선         위도          경도
751       강남         2호선  37.497990  127.027912
741  구로디지털단지         2호선  37.485266  126.901401
721      구파발         3호선  37.636763  126.918821
677       길음         4호선  37.603407  127.025053
777      동대문         1호선  37.571687  127.010930
327       망원         6호선  37.556094  126.910052
670       명동         4호선  37.560989  126.986325
678    미아사거리         4호선  37.613292  127.030053
389       발산         5호선  37.558598  126.837668
744       봉천         2호선  37.482362  126.941892
747       사당         2호선  37.476538  126.981544
684       상계         4호선  37.660878  127.073572
5         서울  수도권 광역급행철도  37.555690  126.972960
753       선릉         2호선  37.504286  127.048203
742      신대방         2호선  37.487462  126.913149
739      신도림         2호선  37.508961  126.891084
743       신림         2호선  37.484201  126.929715
681       쌍문         4호선  37.648627  127.034709
713       안국         3호선  37.576477  126.985443
242       암사         8호

In [11]:
# 어떤 역이 누락됐는지
matched = set(target_df_unique['역사명'])
missing = [s for s in target_stations if s not in matched]
print("누락된 역:", missing)

누락된 역: ['청량리', '회현', '수유', '강변', '남부터미널', '교대', '서울대입구', '광화문', '구의', '잠실', '양재', '미아', '천호']


In [12]:
import re

# 역명 정규화 함수
def clean_name_final(text):
    if pd.isna(text): return ""
    text = str(text)
    text = re.sub(r'\(.*\)', '', text)   # 괄호 제거
    if text.endswith('역') and len(text) > 2:
        text = text[:-1]                 # '역' 제거
    text = text.replace('환승', '')
    text = re.sub(r'[^가-힣a-zA-Z0-9]', '', text)
    return text.strip()

# 양쪽 다 정규화 키 생성
station_df['역명_key'] = station_df['역사명'].apply(clean_name_final)
priority_df['역명_key'] = priority_df['지하철역'].apply(clean_name_final)

target_stations_key = priority_df['역명_key'].tolist()

# 정규화 키로 필터링
target_df = station_df[station_df['역명_key'].isin(target_stations_key)].copy()

# 호선 우선순위로 중복 제거
target_df['호선_우선순위'] = target_df['호선'].str.match(r'^\d').astype(int)
target_df_unique = (
    target_df
    .sort_values('호선_우선순위', ascending=False)
    .drop_duplicates(subset='역명_key', keep='first')
    .copy()
)

print(f"매칭된 역 수: {len(target_df_unique)}개")

# 누락 확인
matched = set(target_df_unique['역명_key'])
missing = [s for s in target_stations_key if s not in matched]
print("누락된 역:", missing)

매칭된 역 수: 49개
누락된 역: []


In [13]:
# 자치구 파일로 자동 매핑
borough_df['해당역(호선)'] = borough_df['해당역(호선)'].str.split(',')
df_mapping = borough_df.explode('해당역(호선)')
df_mapping['역명_key'] = df_mapping['해당역(호선)'].apply(clean_name_final)
df_mapping_unique = df_mapping.drop_duplicates(subset='역명_key')

# target_df_unique에 자치구 merge
target_df_unique = target_df_unique.merge(
    df_mapping_unique[['역명_key', '자치구']],
    on='역명_key',
    how='left'
).rename(columns={'자치구': '구'})

# 매핑 확인
print(f"구 매핑 실패한 역: {target_df_unique['구'].isna().sum()}개")
print(target_df_unique[['역사명', '역명_key', '구']])

구 매핑 실패한 역: 0개
             역사명   역명_key     구
0       수유(강북구청)       수유   강북구
1             홍제       홍제  서대문구
2            신대방      신대방   동작구
3        구로디지털단지  구로디지털단지   구로구
4            신도림      신도림   구로구
5           홍대입구     홍대입구   마포구
6            구파발      구파발   은평구
7            연신내      연신내   은평구
8             안국       안국   종로구
9             봉천       봉천   관악구
10          종로3가     종로3가   종로구
11         을지로3가    을지로3가    중구
12    교대(법원.검찰청)       교대   서초구
13  남부터미널(예술의전당)    남부터미널   서초구
14      양재(서초구청)       양재   서초구
15            상계       상계   노원구
16            신림       신림   관악구
17   서울대입구(관악구청)    서울대입구   관악구
18            쌍문       쌍문   도봉구
19            종각       종각   종로구
20          종로5가     종로5가   종로구
21           동대문      동대문   종로구
22           제기동      제기동  동대문구
23  청량리(서울시립대입구)      청량리  동대문구
24      구의(광진구청)       구의   광진구
25            사당       사당   동작구
26    강변(동서울터미널)       강변   광진구
27          잠실나루     잠실나루   송파구
28      잠실(송파구청)       잠실   송파구
29            선릉       선릉

In [14]:
# 지축역 구 수동 패치 (borough_df에 미포함)
target_df_unique.loc[target_df_unique['역명_key'] == '지축', '구'] = '은평구'

# 최종 확인
print(f"구 매핑 실패한 역: {target_df_unique['구'].isna().sum()}개")
print(target_df_unique[['역명_key', '구']].sort_values('구'))

구 매핑 실패한 역: 0개
     역명_key     구
30       강남   강남구
29       선릉   강남구
43       암사   강동구
34       천호   강동구
0        수유   강북구
40       미아   강북구
48    미아사거리   강북구
33       화곡   강서구
42       발산   강서구
41      우장산   강서구
9        봉천   관악구
17    서울대입구   관악구
16       신림   관악구
26       강변   광진구
24       구의   광진구
4       신도림   구로구
3   구로디지털단지   구로구
15       상계   노원구
35       하계   노원구
18       쌍문   도봉구
31       창동   도봉구
22      제기동  동대문구
23      청량리  동대문구
38      장한평  동대문구
25       사당   동작구
2       신대방   동작구
5      홍대입구   마포구
36       망원   마포구
1        홍제  서대문구
14       양재   서초구
12       교대   서초구
13    남부터미널   서초구
47       길음   성북구
27     잠실나루   송파구
28       잠실   송파구
32       서울   용산구
6       구파발   은평구
37       응암   은평구
7       연신내   은평구
21      동대문   종로구
20     종로5가   종로구
39      광화문   종로구
19       종각   종로구
10     종로3가   종로구
8        안국   종로구
46       혜화   종로구
11    을지로3가    중구
44       회현    중구
45       명동    중구


In [15]:
# NaN인 역 찾기
print(target_df_unique[target_df_unique['구'].isna()][['역사명', '역명_key', '구']])

Empty DataFrame
Columns: [역사명, 역명_key, 구]
Index: []


In [16]:
# 총신대입구 수동 패치
target_df_unique.loc[target_df_unique['역명_key'] == '총신대입구', '구'] = '동작구'

# 최종 확인
print(f"구 매핑 실패한 역: {target_df_unique['구'].isna().sum()}개")

구 매핑 실패한 역: 0개


In [17]:
# 구별 요양기관 수 집계
nursing_seoul['구'] = nursing_seoul['시도 시군구 법정동명'].str.split().str[1]
nursing_by_gu = nursing_seoul.groupby('구').size().reset_index(name='요양기관수')

# 구별 장애인복지관 수 집계
welfare_by_gu = welfare_seoul.groupby('시군구').size().reset_index(name='장애인복지관수')
welfare_by_gu = welfare_by_gu.rename(columns={'시군구': '구'})

print(nursing_by_gu.head())
print(welfare_by_gu.head())

     구  요양기관수
0  강남구    123
1  강동구    177
2  강북구    174
3  강서구    288
4  관악구    211
     구  장애인복지관수
0  강남구        6
1  강동구        4
2  강북구        1
3  강서구        3
4  관악구        2


In [18]:
# 복지시설 수 전체 merge
target_with_welfare = target_df_unique.merge(nursing_by_gu, on='구', how='left') \
                                       .merge(welfare_by_gu, on='구', how='left') \
                                       .merge(hospital_by_gu, on='구', how='left')

# 결측치 0으로 채우기
for col in ['요양기관수', '장애인복지관수', '병원수']:
    target_with_welfare[col] = target_with_welfare[col].fillna(0).astype(int)

# 최종 확인
print(f"최종 shape: {target_with_welfare.shape}")
print(target_with_welfare[['역명_key', '구', '요양기관수', '장애인복지관수', '병원수']])

최종 shape: (49, 11)
     역명_key     구  요양기관수  장애인복지관수  병원수
0        수유   강북구    174        1    6
1        홍제  서대문구    116        2    7
2       신대방   동작구    140        3    3
3   구로디지털단지   구로구    177        2    9
4       신도림   구로구    177        2    9
5      홍대입구   마포구     91        1    3
6       구파발   은평구    218        2    7
7       연신내   은평구    218        2    7
8        안국   종로구     60        1    5
9        봉천   관악구    211        2    3
10     종로3가   종로구     60        1    5
11    을지로3가    중구     38        1    1
12       교대   서초구     91        2    6
13    남부터미널   서초구     91        2    6
14       양재   서초구     91        2    6
15       상계   노원구    250        6   10
16       신림   관악구    211        2    3
17    서울대입구   관악구    211        2    3
18       쌍문   도봉구    242        1    7
19       종각   종로구     60        1    5
20     종로5가   종로구     60        1    5
21      동대문   종로구     60        1    5
22      제기동  동대문구    183        2    9
23      청량리  동대문구    183        2    9
24    

In [19]:
# priority_df에서 필요한 컬럼만 선택해서 merge
final_df = target_with_welfare.merge(
    priority_df[['역명_key', '무임승하차비중', '무임승하차', '시설부하지수', 'TOTAL_W']],
    on='역명_key',
    how='left'
)

# merge 결과 확인
print(f"shape: {final_df.shape}")
print(f"결측치 확인:\n{final_df[['무임승하차비중', '시설부하지수']].isna().sum()}")

shape: (49, 15)
결측치 확인:
무임승하차비중    0
시설부하지수     0
dtype: int64


In [20]:
# TOTAL_W → TOTAL로 컬럼명 변경
final_df = final_df.rename(columns={'TOTAL_W': 'TOTAL'})

# 복지시설합계 컬럼 생성
final_df['복지시설합계'] = final_df['요양기관수'] + final_df['장애인복지관수'] + final_df['병원수']

# 저장
final_df.to_csv('priority_final.csv', index=False, encoding='utf-8-sig')
print(f"저장 완료: {len(final_df)}개 역")
print(final_df[['역명_key', 'TOTAL', '복지시설합계']].head())

저장 완료: 49개 역
    역명_key  TOTAL  복지시설합계
0       수유     16     181
1       홍제     12     125
2      신대방     20     146
3  구로디지털단지     20     188
4      신도림     25     188


In [21]:
final_df.to_csv('priority_district.csv', index=False, encoding='utf-8-sig')
print(f"저장 완료: {len(final_df)}개 역")

저장 완료: 49개 역
